In [0]:
%sql
create schema if not exists 02_silver_catalog.silver 

In [0]:
bronze_schema = "01_bronze_catalog.bronze_azure_blob."
silver_schema = "02_silver_catalog.silver."

### Customer Survey Table

In [0]:
df_customer_survey = spark.table(bronze_schema+"customer_survey")
customer_survey = df_customer_survey.drop("_line","_file","_fivetran_synced","_modified")
customer_survey.display()

In [0]:
from pyspark.sql.functions import col, when, to_date, year, month, to_timestamp, date_format

# Transform and enrich customer_survey by splitting date and time
customer_survey = customer_survey.withColumns(
    {
        "responded_flag": when(col("responded_flag") == "true", 1).otherwise(0),
        "survey_sent_date": to_date(col("survey_sent_date")),
        "survey_sent_time": date_format(to_timestamp(col("survey_sent_date")), "HH:mm:ss"),
        "survey_response_date": to_date(col("survey_response_date")),
        "survey_response_time": date_format(to_timestamp(col("survey_response_date")), "HH:mm:ss"),
    }
)

display(customer_survey)

In [0]:
customer_survey.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"customer_survey")

### Estimate Table


In [0]:
df_estimate = spark.table(bronze_schema+"estimate")
estimate = df_estimate.drop("_line","_file","_fivetran_synced","_modified","version_no")
estimate.display()

In [0]:
from pyspark.sql.functions import col, to_date, year, month, lit

estimate = estimate.withColumns(
    {
        "created_at": to_date(col("created_at")),
        "created_at_time": date_format(to_timestamp(col("created_at")), "HH:mm:ss"),
        "estimate_amount": col("estimate_amount").cast("double")
    }
)
display(estimate)

In [0]:
estimate.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"estimate")

### Invoice Table

In [0]:
df_invoice = spark.table(bronze_schema+"invoice")
invoice = df_invoice.drop("_line","_file","_fivetran_synced","_modified")
invoice.display()

In [0]:
from pyspark.sql.functions import col, to_date, year, month, date_format, to_timestamp

# Transform invoice table for downstream KPIs
invoice = invoice.withColumns(
    {
        "invoice_date_time": date_format(to_timestamp(col("invoice_date")), "HH:mm:ss"),
        "invoice_date": to_date(col("invoice_date")),
        "invoice_amount": col("invoice_amount").cast("double"),
    }
)

display(invoice)

In [0]:
invoice.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"invoice")

### NS Budget Table

In [0]:
df_ns_budget = spark.table(bronze_schema+"ns_budget")
ns_budget = df_ns_budget.drop("_line","_file","_fivetran_synced","_modified")
ns_budget.display()

In [0]:
from pyspark.sql.functions import col, to_date, year, month

# Transform ns_budget for downstream KPIs
ns_budget = ns_budget.withColumns(
    {
        "budget_amount": col("budget_amount").cast("double"),
        "month": to_date(col("month")),
        "budget_year": year(to_date(col("month"))),
        "budget_month": month(to_date(col("month")))
    }
)

display(ns_budget)

In [0]:
ns_budget.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"ns_budget")

### Order Table

In [0]:
df_order = spark.table(bronze_schema+"order")
order = df_order.drop("_line","_file","_fivetran_synced","_modified")
order.display()